In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


# ---------------------------------------------------------
# 1. FIND PROJECT ROOT
# ---------------------------------------------------------

ROOT = Path.cwd().parents[1]


# ---------------------------------------------------------
# 2. DEFINE DATA PATHS
# ---------------------------------------------------------

PROCESSED = ROOT / "data" / "processed"

DATABASE_DIR = ROOT / "data" / "database"

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 3. DATABASE FILE
# ---------------------------------------------------------

DB_PATH = DATABASE_DIR / "controlsight.db"


print("Project root:")
print(ROOT)

print("\nProcessed data folder:")
print(PROCESSED)

print("\nDatabase path:")
print(DB_PATH)

Project root:
/Users/ansh/Desktop/ControlSight

Processed data folder:
/Users/ansh/Desktop/ControlSight/data/processed

Database path:
/Users/ansh/Desktop/ControlSight/data/database/controlsight.db


In [2]:
projects = pd.read_csv(
    PROCESSED / "projects_clean.csv"
)

vendors = pd.read_csv(
    PROCESSED / "vendors_clean.csv"
)

packages = pd.read_csv(
    PROCESSED / "cost_packages_clean.csv"
)

costs = pd.read_csv(
    PROCESSED / "cost_monthly_clean.csv"
)

progress = pd.read_csv(
    PROCESSED / "progress_updates_clean.csv"
)

schedule = pd.read_csv(
    PROCESSED / "schedule_activities_clean.csv"
)

procurement = pd.read_csv(
    PROCESSED / "procurement_packages_clean.csv"
)

data_quality = pd.read_csv(
    PROCESSED / "data_quality_issues.csv"
)

print("Processed datasets loaded successfully.")

Processed datasets loaded successfully.


In [3]:
datasets = {
    "projects": projects,
    "vendors": vendors,
    "packages": packages,
    "costs": costs,
    "progress": progress,
    "schedule": schedule,
    "procurement": procurement,
    "data_quality": data_quality,
}

for name, df in datasets.items():
    print(
        f"{name:20s}"
        f"{len(df):5d} rows x "
        f"{len(df.columns):2d} columns"
    )

projects                8 rows x 10 columns
vendors                30 rows x  4 columns
packages               96 rows x  5 columns
costs                1821 rows x 17 columns
progress              151 rows x  8 columns
schedule              318 rows x 16 columns
procurement           115 rows x 16 columns
data_quality           13 rows x  6 columns


In [4]:
# ---------------------------------------------------------
# CREATE SQLITE DATABASE CONNECTION
# ---------------------------------------------------------

connection = sqlite3.connect(DB_PATH)

print("Connected to SQLite database successfully.")

Connected to SQLite database successfully.


In [5]:
# ---------------------------------------------------------
# TURN ON FOREIGN KEY SUPPORT
# ---------------------------------------------------------

connection.execute("PRAGMA foreign_keys = ON;")

print("Foreign key support enabled.")

Foreign key support enabled.


In [6]:
# ---------------------------------------------------------
# CREATE PROJECT DIMENSION TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS dim_project (
    project_code TEXT PRIMARY KEY,
    project_name TEXT NOT NULL,
    project_type TEXT,
    region TEXT,
    start_date TEXT,
    planned_end_date TEXT,
    approved_budget REAL,
    project_manager TEXT,
    status TEXT,
    currency TEXT
);
""")

connection.commit()

print("dim_project table created successfully.")

dim_project table created successfully.


In [7]:
# ---------------------------------------------------------
# INSERT PROJECT DATA INTO dim_project
# ---------------------------------------------------------

projects.to_sql(
    "dim_project",
    connection,
    if_exists="append",
    index=False
)

print("Project data inserted into dim_project.")

Project data inserted into dim_project.


In [8]:
project_check = pd.read_sql_query(
    """
    SELECT *
    FROM dim_project;
    """,
    connection
)

project_check

,project_code,project_name,project_type,region,start_date,planned_end_date,approved_budget,project_manager,status,currency
0,PRJ-001,Orion Commercial Tower,Commercial Tower,London,2025-01-01,2026-12-31,28000000.0,A. Khan,Active,GBP
1,PRJ-002,Atlas Data Centre,Data Centre,Manchester,2025-03-01,2027-02-28,42000000.0,M. Patel,Active,GBP
2,PRJ-003,Riverside Residential,Residential Development,Birmingham,2025-01-15,2026-10-31,19000000.0,S. Morgan,Active,GBP
3,PRJ-004,NorthLink Infrastructure,Infrastructure,Leeds,2024-11-01,2027-06-30,55000000.0,L. Evans,Active,GBP
4,PRJ-005,Phoenix Industrial Facility,Industrial Facility,Sheffield,2025-06-01,2027-03-31,36000000.0,R. Jones,Active,GBP
5,PRJ-006,St Anne Hospital Expansion,Hospital Expansion,Liverpool,2025-02-01,2026-11-30,31000000.0,D. Clarke,Active,GBP
6,PRJ-007,WestGate Logistics Hub,Logistics Hub,Bristol,2025-08-01,2027-01-31,24000000.0,N. Taylor,Active,GBP
7,PRJ-008,Helios Energy Facility,Energy Facility,Newcastle,2025-04-01,2027-05-31,48000000.0,P. Singh,Active,GBP


In [9]:
print(
    "Rows in dim_project:",
    len(project_check)
)

Rows in dim_project: 8


In [10]:
# ---------------------------------------------------------
# CREATE VENDOR DIMENSION TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS dim_vendor (
    vendor_id TEXT PRIMARY KEY,
    vendor_name TEXT NOT NULL,
    vendor_category TEXT,
    country TEXT
);
""")

connection.commit()

print("dim_vendor table created successfully.")

dim_vendor table created successfully.


In [11]:
# ---------------------------------------------------------
# INSERT VENDOR DATA
# ---------------------------------------------------------

vendors.to_sql(
    "dim_vendor",
    connection,
    if_exists="append",
    index=False
)

print("Vendor data inserted into dim_vendor.")

Vendor data inserted into dim_vendor.


In [12]:
vendor_check = pd.read_sql_query(
    """
    SELECT *
    FROM dim_vendor;
    """,
    connection
)

vendor_check

,vendor_id,vendor_name,vendor_category,country
0,V-001,Vendor 01 Ltd,HVAC,Germany
1,V-002,Vendor 02 Ltd,Steel,UK
2,V-003,Vendor 03 Ltd,Mechanical,UK
3,V-004,Vendor 04 Ltd,Steel,UK
4,V-005,Vendor 05 Ltd,Steel,UK
5,V-006,Vendor 06 Ltd,Electrical,France
6,V-007,Vendor 07 Ltd,Mechanical,UK
7,V-008,Vendor 08 Ltd,Mechanical,UK
8,V-009,Vendor 09 Ltd,Mechanical,UK
9,V-010,Vendor 10 Ltd,Steel,UK


In [13]:
print(
    "Rows in dim_vendor:",
    len(vendor_check)
)

Rows in dim_vendor: 30


In [14]:
# ---------------------------------------------------------
# CREATE PACKAGE DIMENSION TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS dim_package (
    package_code TEXT PRIMARY KEY,
    project_code TEXT NOT NULL,
    package_name TEXT NOT NULL,
    cost_category TEXT,
    package_budget REAL,

    FOREIGN KEY (project_code)
        REFERENCES dim_project(project_code)
);
""")

connection.commit()

print("dim_package table created successfully.")

dim_package table created successfully.


In [15]:
# ---------------------------------------------------------
# INSERT PACKAGE DATA
# ---------------------------------------------------------

packages.to_sql(
    "dim_package",
    connection,
    if_exists="append",
    index=False
)

print("Package data inserted into dim_package.")

Package data inserted into dim_package.


In [16]:
package_check = pd.read_sql_query(
    """
    SELECT *
    FROM dim_package;
    """,
    connection
)

package_check

,package_code,project_code,package_name,cost_category,package_budget
0,PRJ-001-PKG-01,PRJ-001,Civil Package 01,Civil,6493032.57
1,PRJ-001-PKG-02,PRJ-001,Structural Package 02,Structural,2774043.19
2,PRJ-001-PKG-03,PRJ-001,Mechanical Package 03,Mechanical,5214593.51
3,PRJ-001-PKG-04,PRJ-001,Electrical Package 04,Electrical,4186688.73
4,PRJ-001-PKG-05,PRJ-001,HVAC Package 05,HVAC,1693625.14
...,...,...,...,...,...
91,PRJ-008-PKG-08,PRJ-008,Structural Package 08,Structural,979846.30
92,PRJ-008-PKG-09,PRJ-008,Mechanical Package 09,Mechanical,2322958.67
93,PRJ-008-PKG-10,PRJ-008,Electrical Package 10,Electrical,1209112.84
94,PRJ-008-PKG-11,PRJ-008,HVAC Package 11,HVAC,1132495.65


In [17]:
print(
    "Rows in dim_package:",
    len(package_check)
)

Rows in dim_package: 96


In [18]:
# ---------------------------------------------------------
# CREATE DATE DIMENSION DATA
# ---------------------------------------------------------

date_range = pd.date_range(
    start="2024-01-01",
    end="2027-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "date": date_range
})

dim_date["date_key"] = (
    dim_date["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = (
    dim_date["date"]
    .dt.year
)

dim_date["month_number"] = (
    dim_date["date"]
    .dt.month
)

dim_date["month_name"] = (
    dim_date["date"]
    .dt.month_name()
)

dim_date["quarter"] = (
    "Q"
    + dim_date["date"]
    .dt.quarter
    .astype(str)
)

dim_date["year_month"] = (
    dim_date["date"]
    .dt.strftime("%Y-%m")
)

dim_date["date"] = (
    dim_date["date"]
    .dt.strftime("%Y-%m-%d")
)

dim_date.head()

,date,date_key,year,month_number,month_name,quarter,year_month
0,2024-01-01,20240101,2024,1,January,Q1,2024-01
1,2024-01-02,20240102,2024,1,January,Q1,2024-01
2,2024-01-03,20240103,2024,1,January,Q1,2024-01
3,2024-01-04,20240104,2024,1,January,Q1,2024-01
4,2024-01-05,20240105,2024,1,January,Q1,2024-01


In [19]:
# ---------------------------------------------------------
# CREATE dim_date TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY,
    date TEXT NOT NULL,
    year INTEGER,
    month_number INTEGER,
    month_name TEXT,
    quarter TEXT,
    year_month TEXT
);
""")

connection.commit()

print("dim_date table created successfully.")

dim_date table created successfully.


In [20]:
dim_date.to_sql(
    "dim_date",
    connection,
    if_exists="append",
    index=False
)

print("Date data inserted into dim_date.")

Date data inserted into dim_date.


In [21]:
date_check = pd.read_sql_query(
    """
    SELECT *
    FROM dim_date
    LIMIT 10;
    """,
    connection
)

date_check

,date_key,date,year,month_number,month_name,quarter,year_month
0,20240101,2024-01-01,2024,1,January,Q1,2024-01
1,20240102,2024-01-02,2024,1,January,Q1,2024-01
2,20240103,2024-01-03,2024,1,January,Q1,2024-01
3,20240104,2024-01-04,2024,1,January,Q1,2024-01
4,20240105,2024-01-05,2024,1,January,Q1,2024-01
5,20240106,2024-01-06,2024,1,January,Q1,2024-01
6,20240107,2024-01-07,2024,1,January,Q1,2024-01
7,20240108,2024-01-08,2024,1,January,Q1,2024-01
8,20240109,2024-01-09,2024,1,January,Q1,2024-01
9,20240110,2024-01-10,2024,1,January,Q1,2024-01


In [22]:
# ---------------------------------------------------------
# CREATE FACT COST TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS fact_cost (
    cost_record_id TEXT PRIMARY KEY,
    project_code TEXT NOT NULL,
    package_code TEXT NOT NULL,
    date_key INTEGER NOT NULL,
    cost_category TEXT,
    planned_cost REAL,
    actual_cost REAL,
    committed_cost REAL,
    forecast_at_completion REAL,
    package_budget REAL,
    cost_variance REAL,
    cost_variance_pct REAL,
    forecast_variance REAL,
    forecast_variance_pct REAL,
    cost_anomaly_flag INTEGER,
    currency TEXT,
    source_system TEXT,

    FOREIGN KEY (project_code)
        REFERENCES dim_project(project_code),

    FOREIGN KEY (package_code)
        REFERENCES dim_package(package_code),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key)
);
""")

connection.commit()

print("fact_cost table created successfully.")

fact_cost table created successfully.


In [25]:
# ---------------------------------------------------------
# PREPARE FACT COST DATA
# ---------------------------------------------------------

fact_cost = costs.copy()

fact_cost["date_key"] = (
    pd.to_datetime(fact_cost["report_month"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_cost = fact_cost[
    [
        "cost_record_id",
        "project_code",
        "package_code",
        "date_key",
        "cost_category",
        "planned_cost",
        "actual_cost",
        "committed_cost",
        "forecast_at_completion",
        "package_budget",
        "cost_variance",
        "cost_variance_pct",
        "forecast_variance",
        "forecast_variance_pct",
        "cost_anomaly_flag",
        "currency",
        "source_system",
    ]
]

fact_cost.head()

,cost_record_id,project_code,package_code,date_key,cost_category,planned_cost,actual_cost,committed_cost,forecast_at_completion,package_budget,cost_variance,cost_variance_pct,forecast_variance,forecast_variance_pct,cost_anomaly_flag,currency,source_system
0,C-000001,PRJ-001,PRJ-001-PKG-01,20250131,Civil,200572.03,191419.61,206308.06,6745303.66,6493032.57,9152.42,4.563159,-252271.09,-3.885258,False,GBP,ERP_COST
1,C-000002,PRJ-001,PRJ-001-PKG-02,20250131,Structural,85691.16,85236.76,96286.32,2788355.37,2774043.19,454.40,0.530276,-14312.18,-0.515932,False,GBP,ERP_COST
2,C-000003,PRJ-001,PRJ-001-PKG-03,20250131,Mechanical,161080.60,167524.71,167524.71,5433249.95,5214593.51,-6444.11,-4.000550,-218656.44,-4.193164,False,GBP,ERP_COST
3,C-000004,PRJ-001,PRJ-001-PKG-04,20250131,Electrical,129328.27,126722.65,147595.69,4204065.36,4186688.73,2605.62,2.014734,-17376.63,-0.415045,False,GBP,ERP_COST
4,C-000005,PRJ-001,PRJ-001-PKG-05,20250131,HVAC,52316.67,47323.64,60821.26,1775476.36,1693625.14,4993.03,9.543860,-81851.22,-4.832901,False,GBP,ERP_COST


In [26]:
fact_cost.to_sql(
    "fact_cost",
    connection,
    if_exists="append",
    index=False
)

print("Cost data inserted into fact_cost.")

Cost data inserted into fact_cost.


In [27]:
cost_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM fact_cost;
    """,
    connection
)

cost_check

,row_count
0,1821


In [30]:
progress = pd.read_csv(
    PROCESSED / "progress_updates_clean.csv"
)

fact_progress = progress.copy()

fact_progress["date_key"] = (
    pd.to_datetime(
        fact_progress["report_month"]
    )
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_progress = fact_progress[
    [
        "project_code",
        "date_key",
        "planned_progress_pct",
        "actual_progress_pct",
        "progress_variance_pct",
        "behind_schedule_flag",
        "reported_by",
        "source_system",
    ]
]

fact_progress.head()

,project_code,date_key,planned_progress_pct,actual_progress_pct,progress_variance_pct,behind_schedule_flag,reported_by,source_system
0,PRJ-001,20250131,4.17,4.17,0.00,False,Site Team,P6_EXPORT
1,PRJ-001,20250228,8.33,11.86,3.53,False,Project Controls,P6_EXPORT
2,PRJ-001,20250331,12.50,12.97,0.47,False,Project Controls,P6_EXPORT
3,PRJ-001,20250430,16.67,15.94,-0.73,False,Site Team,P6_EXPORT
4,PRJ-001,20250531,20.83,20.36,-0.47,False,Site Team,P6_EXPORT


In [31]:
connection.execute("""
CREATE TABLE IF NOT EXISTS fact_progress (
    progress_id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_code TEXT NOT NULL,
    date_key INTEGER NOT NULL,
    planned_progress_pct REAL,
    actual_progress_pct REAL,
    progress_variance_pct REAL,
    behind_schedule_flag INTEGER,
    reported_by TEXT,
    source_system TEXT,

    FOREIGN KEY (project_code)
        REFERENCES dim_project(project_code),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key)
);
""")

connection.commit()

print("fact_progress table created successfully.")

fact_progress table created successfully.


In [32]:
fact_progress.to_sql(
    "fact_progress",
    connection,
    if_exists="append",
    index=False
)

print("Progress data inserted into fact_progress.")

Progress data inserted into fact_progress.


In [33]:
progress_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM fact_progress;
    """,
    connection
)

progress_check

,row_count
0,151


In [34]:
# ---------------------------------------------------------
# PREPARE FACT SCHEDULE DATA
# ---------------------------------------------------------

fact_schedule = schedule.copy()

fact_schedule["planned_start_key"] = (
    pd.to_datetime(fact_schedule["planned_start"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_schedule["planned_finish_key"] = (
    pd.to_datetime(fact_schedule["planned_finish"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_schedule["actual_start_key"] = (
    pd.to_datetime(
        fact_schedule["actual_start"],
        errors="coerce"
    )
    .dt.strftime("%Y%m%d")
)

fact_schedule["actual_finish_key"] = (
    pd.to_datetime(
        fact_schedule["actual_finish"],
        errors="coerce"
    )
    .dt.strftime("%Y%m%d")
)

fact_schedule.head()

,activity_id,project_code,wbs_code,activity_name,discipline,planned_start,planned_finish,actual_start,actual_finish,percent_complete,status,is_critical,source_system,start_delay_days,finish_delay_days,delayed_activity_flag,planned_start_key,planned_finish_key,actual_start_key,actual_finish_key
0,PRJ-001-ACT-001,PRJ-001,WBS-01,Design approval,Civil,2025-01-03,2025-02-06,2025-01-04,2025-02-14,100.0,Complete,False,P6_EXPORT,1.0,8.0,True,20250103,20250206,20250104,20250214
1,PRJ-001-ACT-002,PRJ-001,WBS-01,Site mobilisation,Structural,2025-01-16,2025-02-19,2025-01-19,2025-02-22,100.0,Complete,False,P6_EXPORT,3.0,3.0,True,20250116,20250219,20250119,20250222
2,PRJ-001-ACT-003,PRJ-001,WBS-01,Groundworks,Mechanical,2025-01-31,2025-03-13,2025-02-07,2025-03-17,100.0,Complete,False,P6_EXPORT,7.0,4.0,True,20250131,20250313,20250207,20250317
3,PRJ-001-ACT-004,PRJ-001,WBS-01,Structural frame,Electrical,2025-02-10,2025-03-13,2025-02-10,2025-03-13,100.0,Complete,False,P6_EXPORT,0.0,0.0,False,20250210,20250313,20250210,20250313
4,PRJ-001-ACT-005,PRJ-001,WBS-01,MEP installation,HVAC,2025-02-20,2025-04-03,2025-02-20,2025-04-03,100.0,Complete,False,P6_EXPORT,0.0,0.0,False,20250220,20250403,20250220,20250403


In [35]:
fact_schedule["actual_start_key"] = pd.to_numeric(
    fact_schedule["actual_start_key"],
    errors="coerce"
).astype("Int64")

fact_schedule["actual_finish_key"] = pd.to_numeric(
    fact_schedule["actual_finish_key"],
    errors="coerce"
).astype("Int64")

In [36]:
fact_schedule = fact_schedule[
    [
        "activity_id",
        "project_code",
        "wbs_code",
        "activity_name",
        "discipline",
        "planned_start_key",
        "planned_finish_key",
        "actual_start_key",
        "actual_finish_key",
        "percent_complete",
        "status",
        "is_critical",
        "start_delay_days",
        "finish_delay_days",
        "delayed_activity_flag",
        "source_system",
    ]
]

fact_schedule.head()

,activity_id,project_code,wbs_code,activity_name,discipline,planned_start_key,planned_finish_key,actual_start_key,actual_finish_key,percent_complete,status,is_critical,start_delay_days,finish_delay_days,delayed_activity_flag,source_system
0,PRJ-001-ACT-001,PRJ-001,WBS-01,Design approval,Civil,20250103,20250206,20250104,20250214,100.0,Complete,False,1.0,8.0,True,P6_EXPORT
1,PRJ-001-ACT-002,PRJ-001,WBS-01,Site mobilisation,Structural,20250116,20250219,20250119,20250222,100.0,Complete,False,3.0,3.0,True,P6_EXPORT
2,PRJ-001-ACT-003,PRJ-001,WBS-01,Groundworks,Mechanical,20250131,20250313,20250207,20250317,100.0,Complete,False,7.0,4.0,True,P6_EXPORT
3,PRJ-001-ACT-004,PRJ-001,WBS-01,Structural frame,Electrical,20250210,20250313,20250210,20250313,100.0,Complete,False,0.0,0.0,False,P6_EXPORT
4,PRJ-001-ACT-005,PRJ-001,WBS-01,MEP installation,HVAC,20250220,20250403,20250220,20250403,100.0,Complete,False,0.0,0.0,False,P6_EXPORT


In [37]:
# ---------------------------------------------------------
# CREATE FACT SCHEDULE TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS fact_schedule (
    activity_id TEXT PRIMARY KEY,
    project_code TEXT NOT NULL,
    wbs_code TEXT,
    activity_name TEXT,
    discipline TEXT,
    planned_start_key INTEGER,
    planned_finish_key INTEGER,
    actual_start_key INTEGER,
    actual_finish_key INTEGER,
    percent_complete REAL,
    status TEXT,
    is_critical INTEGER,
    start_delay_days REAL,
    finish_delay_days REAL,
    delayed_activity_flag INTEGER,
    source_system TEXT,

    FOREIGN KEY (project_code)
        REFERENCES dim_project(project_code),

    FOREIGN KEY (planned_start_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (planned_finish_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (actual_start_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (actual_finish_key)
        REFERENCES dim_date(date_key)
);
""")

connection.commit()

print("fact_schedule table created successfully.")

fact_schedule table created successfully.


In [38]:
fact_schedule.to_sql(
    "fact_schedule",
    connection,
    if_exists="append",
    index=False
)

print("Schedule data inserted into fact_schedule.")

Schedule data inserted into fact_schedule.


In [39]:
schedule_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM fact_schedule;
    """,
    connection
)

schedule_check

,row_count
0,318


In [40]:
# ---------------------------------------------------------
# PREPARE FACT PROCUREMENT DATA
# ---------------------------------------------------------

fact_procurement = procurement.copy()

# Convert procurement dates into date keys
for source_col, key_col in [
    ("rfq_date", "rfq_date_key"),
    ("po_date", "po_date_key"),
    ("planned_delivery_date", "planned_delivery_key"),
    ("actual_delivery_date", "actual_delivery_key"),
]:
    fact_procurement[key_col] = (
        pd.to_datetime(
            fact_procurement[source_col],
            errors="coerce"
        )
        .dt.strftime("%Y%m%d")
    )

# Convert to nullable integers because some dates can legitimately be missing
for col in [
    "rfq_date_key",
    "po_date_key",
    "planned_delivery_key",
    "actual_delivery_key",
]:
    fact_procurement[col] = pd.to_numeric(
        fact_procurement[col],
        errors="coerce"
    ).astype("Int64")

In [41]:
fact_procurement = fact_procurement[
    [
        "procurement_id",
        "project_code",
        "package_code",
        "vendor_id",
        "category",
        "rfq_date_key",
        "po_date_key",
        "planned_delivery_key",
        "actual_delivery_key",
        "package_value",
        "status",
        "criticality",
        "delivery_delay_days",
        "delayed_delivery_flag",
        "critical_delay_flag",
        "source_system",
    ]
]

fact_procurement.head()

,procurement_id,project_code,package_code,vendor_id,category,rfq_date_key,po_date_key,planned_delivery_key,actual_delivery_key,package_value,status,criticality,delivery_delay_days,delayed_delivery_flag,critical_delay_flag,source_system
0,PO-00001,PRJ-001,PRJ-001-PKG-01,V-004,Civil,20250822,20250909,20251229,20260101,2405511.22,Delivered,Medium,3,True,False,PROCURE_TRACKER
1,PO-00002,PRJ-001,PRJ-001-PKG-02,V-003,Structural,20250328,20250411,20250524,20250529,1031731.65,Delivered,Medium,5,True,False,PROCURE_TRACKER
2,PO-00003,PRJ-001,PRJ-001-PKG-03,V-016,Mechanical,20250728,20250829,20251024,20251025,1945726.28,Delivered,Medium,1,True,False,PROCURE_TRACKER
3,PO-00004,PRJ-001,PRJ-001-PKG-04,V-023,Electrical,20250322,20250422,20250810,20250810,1716307.99,Delivered,Medium,0,False,False,PROCURE_TRACKER
4,PO-00006,PRJ-001,PRJ-001-PKG-05,V-024,HVAC,20250425,20250511,20250904,20250912,1305637.10,Delivered,Medium,8,True,False,PROCURE_TRACKER


In [42]:
# ---------------------------------------------------------
# CREATE FACT PROCUREMENT TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS fact_procurement (
    procurement_id TEXT PRIMARY KEY,
    project_code TEXT NOT NULL,
    package_code TEXT NOT NULL,
    vendor_id TEXT,
    category TEXT,
    rfq_date_key INTEGER,
    po_date_key INTEGER,
    planned_delivery_key INTEGER,
    actual_delivery_key INTEGER,
    package_value REAL,
    status TEXT,
    criticality TEXT,
    delivery_delay_days REAL,
    delayed_delivery_flag INTEGER,
    critical_delay_flag INTEGER,
    source_system TEXT,

    FOREIGN KEY (project_code)
        REFERENCES dim_project(project_code),

    FOREIGN KEY (package_code)
        REFERENCES dim_package(package_code),

    FOREIGN KEY (vendor_id)
        REFERENCES dim_vendor(vendor_id),

    FOREIGN KEY (rfq_date_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (po_date_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (planned_delivery_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (actual_delivery_key)
        REFERENCES dim_date(date_key)
);
""")

connection.commit()

print("fact_procurement table created successfully.")

fact_procurement table created successfully.


In [43]:
fact_procurement.to_sql(
    "fact_procurement",
    connection,
    if_exists="append",
    index=False
)

print("Procurement data inserted into fact_procurement.")

Procurement data inserted into fact_procurement.


In [44]:
procurement_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM fact_procurement;
    """,
    connection
)

procurement_check

,row_count
0,115


In [45]:
# ---------------------------------------------------------
# PREPARE FACT DATA QUALITY DATA
# ---------------------------------------------------------

fact_data_quality = data_quality.copy()

fact_data_quality.head()

,dataset,row_index,issue_type,column,value,message
0,cost_monthly,17,invalid_reference,project_code,PRJ-999,Project code does not exist in projects.csv
1,cost_monthly,80,invalid_value,actual_cost,-25000.0,Actual cost cannot be negative
2,cost_monthly,180,missing_value,package_code,NaN,Package code is missing
3,progress_updates,9,invalid_format,report_month,31/12/2025,Report month is not in YYYY-MM-DD format
4,progress_updates,25,invalid_value,actual_progress_pct,126.0,Actual progress must be between 0 and 100


In [46]:
# ---------------------------------------------------------
# CREATE FACT DATA QUALITY TABLE
# ---------------------------------------------------------

connection.execute("""
CREATE TABLE IF NOT EXISTS fact_data_quality (
    issue_id INTEGER PRIMARY KEY AUTOINCREMENT,
    dataset TEXT,
    row_index INTEGER,
    issue_type TEXT,
    column_name TEXT,
    value TEXT,
    message TEXT
);
""")

connection.commit()

print("fact_data_quality table created successfully.")

fact_data_quality table created successfully.


In [47]:
fact_data_quality = fact_data_quality.rename(
    columns={
        "column": "column_name"
    }
)

In [48]:
fact_data_quality.to_sql(
    "fact_data_quality",
    connection,
    if_exists="append",
    index=False
)

print("Data quality issues inserted into fact_data_quality.")

Data quality issues inserted into fact_data_quality.


In [49]:
data_quality_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM fact_data_quality;
    """,
    connection
)

data_quality_check

,row_count
0,13


In [50]:
# ---------------------------------------------------------
# CHECK ALL DATABASE TABLES
# ---------------------------------------------------------

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
    """,
    connection
)

tables

,name
0,dim_date
1,dim_package
2,dim_project
3,dim_vendor
4,fact_cost
5,fact_data_quality
6,fact_procurement
7,fact_progress
8,fact_schedule
9,sqlite_sequence


In [51]:
table_names = [
    "dim_project",
    "dim_vendor",
    "dim_package",
    "dim_date",
    "fact_cost",
    "fact_progress",
    "fact_schedule",
    "fact_procurement",
    "fact_data_quality",
]

row_counts = []

for table in table_names:

    result = pd.read_sql_query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM {table};
        """,
        connection
    )

    row_counts.append({
        "table": table,
        "row_count": int(result["row_count"].iloc[0])
    })


database_audit = pd.DataFrame(row_counts)

database_audit

,table,row_count
0,dim_project,8
1,dim_vendor,30
2,dim_package,96
3,dim_date,1461
4,fact_cost,1821
5,fact_progress,151
6,fact_schedule,318
7,fact_procurement,115
8,fact_data_quality,13


In [52]:
# ---------------------------------------------------------
# TEST SQL JOIN: PROJECT + COST
# ---------------------------------------------------------

project_cost_summary = pd.read_sql_query(
    """
    SELECT
        p.project_code,
        p.project_name,
        p.region,
        ROUND(SUM(c.planned_cost), 2) AS total_planned_cost,
        ROUND(SUM(c.actual_cost), 2) AS total_actual_cost,
        ROUND(SUM(c.cost_variance), 2) AS total_cost_variance
    FROM fact_cost c
    JOIN dim_project p
        ON c.project_code = p.project_code
    GROUP BY
        p.project_code,
        p.project_name,
        p.region
    ORDER BY
        total_cost_variance ASC;
    """,
    connection
)

project_cost_summary

,project_code,project_name,region,total_planned_cost,total_actual_cost,total_cost_variance
0,PRJ-006,St Anne Hospital Expansion,Liverpool,31968973.29,36138393.25,-4169419.96
1,PRJ-003,Riverside Residential,Birmingham,20241286.41,24382525.02,-4141238.61
2,PRJ-002,Atlas Data Centre,Manchester,38744153.17,41722937.17,-2978784.00
3,PRJ-001,Orion Commercial Tower,London,27558644.71,30496695.01,-2938050.30
4,PRJ-008,Helios Energy Facility,Newcastle,38882149.93,41553331.23,-2671181.30
5,PRJ-005,Phoenix Industrial Facility,Sheffield,30702770.35,32138509.46,-1435739.11
6,PRJ-004,NorthLink Infrastructure,Leeds,46173612.88,45922243.09,251369.79
7,PRJ-007,WestGate Logistics Hub,Bristol,21853425.00,21406361.89,447063.11


In [53]:
# ---------------------------------------------------------
# TEST SQL JOIN: PROCUREMENT + VENDOR + PROJECT
# ---------------------------------------------------------

procurement_summary = pd.read_sql_query(
    """
    SELECT
        p.project_name,
        v.vendor_name,
        fp.status,
        fp.criticality,
        fp.package_value,
        fp.delivery_delay_days
    FROM fact_procurement fp

    JOIN dim_project p
        ON fp.project_code = p.project_code

    JOIN dim_vendor v
        ON fp.vendor_id = v.vendor_id

    ORDER BY
        fp.delivery_delay_days DESC;
    """,
    connection
)

procurement_summary.head(10)

,project_name,vendor_name,status,criticality,package_value,delivery_delay_days
0,Riverside Residential,Vendor 16 Ltd,Delivered,Medium,632012.53,27.0
1,Helios Energy Facility,Vendor 06 Ltd,Delivered,Low,6801480.29,22.0
2,WestGate Logistics Hub,Vendor 10 Ltd,Delivered,Medium,1459657.53,21.0
3,Orion Commercial Tower,Vendor 01 Ltd,Delivered,Medium,54612.84,20.0
4,Atlas Data Centre,Vendor 25 Ltd,Delivered,Medium,416111.27,20.0
5,Phoenix Industrial Facility,Vendor 09 Ltd,Delivered,Medium,4859954.18,20.0
6,Atlas Data Centre,Vendor 28 Ltd,Delivered,Medium,654758.76,19.0
7,St Anne Hospital Expansion,Vendor 28 Ltd,Delivered,Medium,250174.52,19.0
8,Helios Energy Facility,Vendor 03 Ltd,Delivered,Medium,74198.30,19.0
9,St Anne Hospital Expansion,Vendor 17 Ltd,Delivered,Medium,3452701.08,17.0


In [54]:
# ---------------------------------------------------------
# PORTFOLIO EXECUTIVE KPI QUERY
# ---------------------------------------------------------

portfolio_kpis = pd.read_sql_query(
    """
    WITH latest_package_forecast AS (
        SELECT
            package_code,
            MAX(date_key) AS latest_date_key
        FROM fact_cost
        GROUP BY package_code
    ),

    package_forecast AS (
        SELECT
            fc.package_code,
            fc.forecast_at_completion
        FROM fact_cost fc
        JOIN latest_package_forecast lpf
            ON fc.package_code = lpf.package_code
           AND fc.date_key = lpf.latest_date_key
    )

    SELECT
        ROUND(
            (SELECT SUM(approved_budget)
             FROM dim_project),
            2
        ) AS total_budget,

        ROUND(
            (SELECT SUM(actual_cost)
             FROM fact_cost),
            2
        ) AS actual_cost_to_date,

        ROUND(
            (SELECT SUM(committed_cost)
             FROM fact_cost),
            2
        ) AS committed_cost_to_date,

        ROUND(
            (SELECT SUM(forecast_at_completion)
             FROM package_forecast),
            2
        ) AS forecast_at_completion
    ;
    """,
    connection
)

portfolio_kpis

,total_budget,actual_cost_to_date,committed_cost_to_date,forecast_at_completion
0,283000000.0,2.737610e+08,2.892594e+08,2.989639e+08


In [55]:
portfolio_kpis["forecast_variance"] = (
    portfolio_kpis["total_budget"]
    - portfolio_kpis["forecast_at_completion"]
)

portfolio_kpis

,total_budget,actual_cost_to_date,committed_cost_to_date,forecast_at_completion,forecast_variance
0,283000000.0,2.737610e+08,2.892594e+08,2.989639e+08,-15963873.49


In [56]:
# ---------------------------------------------------------
# LATEST PROJECT PROGRESS KPI
# ---------------------------------------------------------

latest_progress = pd.read_sql_query(
    """
    WITH latest_dates AS (
        SELECT
            project_code,
            MAX(date_key) AS latest_date_key
        FROM fact_progress
        GROUP BY project_code
    )

    SELECT
        fp.project_code,
        p.project_name,
        fp.planned_progress_pct,
        fp.actual_progress_pct,
        fp.progress_variance_pct,
        fp.behind_schedule_flag
    FROM fact_progress fp

    JOIN latest_dates ld
        ON fp.project_code = ld.project_code
       AND fp.date_key = ld.latest_date_key

    JOIN dim_project p
        ON fp.project_code = p.project_code

    ORDER BY fp.progress_variance_pct ASC;
    """,
    connection
)

latest_progress

,project_code,project_name,planned_progress_pct,actual_progress_pct,progress_variance_pct,behind_schedule_flag
0,PRJ-003,Riverside Residential,95.45,80.44,-15.01,1
1,PRJ-006,St Anne Hospital Expansion,90.91,76.28,-14.63,1
2,PRJ-002,Atlas Data Centre,79.17,72.00,-7.17,1
3,PRJ-008,Helios Energy Facility,69.23,64.96,-4.27,0
4,PRJ-005,Phoenix Industrial Facility,72.73,70.25,-2.48,0
5,PRJ-004,NorthLink Infrastructure,71.88,71.79,-0.09,0
6,PRJ-001,Orion Commercial Tower,87.50,87.50,0.00,0
7,PRJ-007,WestGate Logistics Hub,77.78,79.10,1.32,0


In [81]:
# ---------------------------------------------------------
# PORTFOLIO PROGRESS KPI
# ---------------------------------------------------------
#
# Portfolio progress is calculated as an approved-budget-
# weighted average of the latest progress snapshot for each
# project.
#
# This avoids giving a small project the same portfolio-level
# influence as a significantly larger project.
#
# Formula:
#
#     SUM(Project Progress × Approved Budget)
#     ---------------------------------------
#              SUM(Approved Budget)
#
# Individual project progress remains unchanged.
# ---------------------------------------------------------

portfolio_progress_input = latest_progress.merge(
    projects[
        [
            "project_code",
            "approved_budget",
        ]
    ],
    on="project_code",
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------
# VALIDATE WEIGHTS
# ---------------------------------------------------------

if portfolio_progress_input["approved_budget"].isna().any():

    missing_projects = (
        portfolio_progress_input.loc[
            portfolio_progress_input[
                "approved_budget"
            ].isna(),
            "project_code",
        ]
        .tolist()
    )

    raise ValueError(
        "Missing approved budget for project(s): "
        + ", ".join(missing_projects)
    )


if (
    portfolio_progress_input["approved_budget"] <= 0
).any():

    invalid_projects = (
        portfolio_progress_input.loc[
            portfolio_progress_input[
                "approved_budget"
            ] <= 0,
            "project_code",
        ]
        .tolist()
    )

    raise ValueError(
        "Invalid approved budget for project(s): "
        + ", ".join(invalid_projects)
    )


total_progress_weight = (
    portfolio_progress_input[
        "approved_budget"
    ].sum()
)


# ---------------------------------------------------------
# BUDGET-WEIGHTED PLANNED PROGRESS
# ---------------------------------------------------------

weighted_planned_progress = (
    (
        portfolio_progress_input[
            "planned_progress_pct"
        ]
        *
        portfolio_progress_input[
            "approved_budget"
        ]
    ).sum()
    /
    total_progress_weight
)


# ---------------------------------------------------------
# BUDGET-WEIGHTED ACTUAL PROGRESS
# ---------------------------------------------------------

weighted_actual_progress = (
    (
        portfolio_progress_input[
            "actual_progress_pct"
        ]
        *
        portfolio_progress_input[
            "approved_budget"
        ]
    ).sum()
    /
    total_progress_weight
)


# ---------------------------------------------------------
# PORTFOLIO PROGRESS SUMMARY
# ---------------------------------------------------------

portfolio_progress_summary = {
    "planned_progress_pct": float(
        round(
            weighted_planned_progress,
            2
        )
    ),

    "actual_progress_pct": float(
        round(
            weighted_actual_progress,
            2
        )
    ),
}


portfolio_progress_summary

{'planned_progress_pct': 78.33, 'actual_progress_pct': 73.71}

In [60]:
project_cost_kpis = pd.read_sql_query(
    """
    WITH latest_package_forecast AS (
        SELECT
            package_code,
            MAX(date_key) AS latest_date_key
        FROM fact_cost
        GROUP BY package_code
    ),

    latest_forecast AS (
        SELECT
            fc.package_code,
            fc.project_code,
            fc.forecast_at_completion
        FROM fact_cost fc

        JOIN latest_package_forecast lpf
            ON fc.package_code = lpf.package_code
           AND fc.date_key = lpf.latest_date_key
    ),

    actuals AS (
        SELECT
            project_code,
            SUM(actual_cost) AS actual_cost_to_date
        FROM fact_cost
        GROUP BY project_code
    ),

    forecasts AS (
        SELECT
            project_code,
            SUM(forecast_at_completion) AS forecast_at_completion
        FROM latest_forecast
        GROUP BY project_code
    )

    SELECT
        p.project_code,
        p.project_name,
        p.approved_budget,

        ROUND(
            a.actual_cost_to_date,
            2
        ) AS actual_cost_to_date,

        ROUND(
            f.forecast_at_completion,
            2
        ) AS forecast_at_completion,

        ROUND(
            p.approved_budget
            - f.forecast_at_completion,
            2
        ) AS forecast_variance

    FROM dim_project p

    LEFT JOIN actuals a
        ON p.project_code = a.project_code

    LEFT JOIN forecasts f
        ON p.project_code = f.project_code

    ORDER BY forecast_variance ASC;
    """,
    connection
)

project_cost_kpis

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98


In [62]:
def assign_health_status(progress_variance):

    if progress_variance >= -3:
        return "Green"

    elif progress_variance >= -8:
        return "Amber"

    else:
        return "Red"


latest_progress["health_status"] = (
    latest_progress["progress_variance_pct"]
    .apply(assign_health_status)
)

latest_progress

,project_code,project_name,planned_progress_pct,actual_progress_pct,progress_variance_pct,behind_schedule_flag,health_status
0,PRJ-003,Riverside Residential,95.45,80.44,-15.01,1,Red
1,PRJ-006,St Anne Hospital Expansion,90.91,76.28,-14.63,1,Red
2,PRJ-002,Atlas Data Centre,79.17,72.00,-7.17,1,Amber
3,PRJ-008,Helios Energy Facility,69.23,64.96,-4.27,0,Amber
4,PRJ-005,Phoenix Industrial Facility,72.73,70.25,-2.48,0,Green
5,PRJ-004,NorthLink Infrastructure,71.88,71.79,-0.09,0,Green
6,PRJ-001,Orion Commercial Tower,87.50,87.50,0.00,0,Green
7,PRJ-007,WestGate Logistics Hub,77.78,79.10,1.32,0,Green


In [63]:
management_scorecard = project_cost_kpis.merge(
    latest_progress[
        [
            "project_code",
            "planned_progress_pct",
            "actual_progress_pct",
            "progress_variance_pct",
            "health_status",
        ]
    ],
    on="project_code",
    how="left"
)

management_scorecard

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green


In [64]:
def assign_management_status(row):

    cost_variance_pct = (
        row["forecast_variance"]
        / row["approved_budget"]
        * 100
    )

    schedule_variance = row["progress_variance_pct"]

    if (
        cost_variance_pct < -8
        or schedule_variance < -8
    ):
        return "Red"

    elif (
        cost_variance_pct < -3
        or schedule_variance < -3
    ):
        return "Amber"

    else:
        return "Green"


management_scorecard["management_status"] = (
    management_scorecard.apply(
        assign_management_status,
        axis=1
    )
)

management_scorecard

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status,management_status
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber,Red
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red,Red
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red,Red
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber,Amber
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green,Amber
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green,Green
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green,Green
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green,Green


In [65]:
management_scorecard["forecast_variance_pct"] = (
    management_scorecard["forecast_variance"]
    / management_scorecard["approved_budget"]
    * 100
).round(2)

management_scorecard

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status,management_status,forecast_variance_pct
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber,Red,-9.32
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red,Red,-13.54
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red,Red,-20.94
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber,Amber,-5.92
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green,Amber,-4.65
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green,Green,-1.53
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green,Green,1.54
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green,Green,1.64


In [66]:
procurement_risk = pd.read_sql_query(
    """
    SELECT
        project_code,

        COUNT(*) AS total_procurement_items,

        SUM(
            CASE
                WHEN delayed_delivery_flag = 1
                THEN 1
                ELSE 0
            END
        ) AS delayed_items,

        SUM(
            CASE
                WHEN critical_delay_flag = 1
                THEN 1
                ELSE 0
            END
        ) AS critical_delays

    FROM fact_procurement

    GROUP BY project_code;
    """,
    connection
)

procurement_risk

,project_code,total_procurement_items,delayed_items,critical_delays
0,PRJ-001,13,9,0
1,PRJ-002,16,14,0
2,PRJ-003,15,12,0
3,PRJ-004,14,9,0
4,PRJ-005,15,12,1
5,PRJ-006,14,14,1
6,PRJ-007,15,7,0
7,PRJ-008,13,11,0


In [67]:
management_scorecard = management_scorecard.merge(
    procurement_risk,
    on="project_code",
    how="left"
)

management_scorecard

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status,management_status,forecast_variance_pct,total_procurement_items,delayed_items,critical_delays
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber,Red,-9.32,13,11,0
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red,Red,-13.54,14,14,1
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red,Red,-20.94,15,12,0
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber,Amber,-5.92,16,14,0
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green,Amber,-4.65,15,12,1
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green,Green,-1.53,13,9,0
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green,Green,1.54,15,7,0
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green,Green,1.64,14,9,0


In [68]:
def assign_final_management_status(row):

    cost_variance_pct = row["forecast_variance_pct"]
    schedule_variance = row["progress_variance_pct"]
    critical_delays = row["critical_delays"]

    if (
        cost_variance_pct < -8
        or schedule_variance < -8
        or critical_delays >= 2
    ):
        return "Red"

    elif (
        cost_variance_pct < -3
        or schedule_variance < -3
        or critical_delays == 1
    ):
        return "Amber"

    else:
        return "Green"


management_scorecard["management_status"] = (
    management_scorecard.apply(
        assign_final_management_status,
        axis=1
    )
)

management_scorecard

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status,management_status,forecast_variance_pct,total_procurement_items,delayed_items,critical_delays
0,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber,Red,-9.32,13,11,0
1,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red,Red,-13.54,14,14,1
2,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red,Red,-20.94,15,12,0
3,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber,Amber,-5.92,16,14,0
4,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green,Amber,-4.65,15,12,1
5,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green,Green,-1.53,13,9,0
6,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green,Green,1.54,15,7,0
7,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green,Green,1.64,14,9,0


In [69]:
# ---------------------------------------------------------
# DATA QUALITY KPI
# ---------------------------------------------------------

data_quality_kpis = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_issues,

        SUM(
            CASE
                WHEN issue_type = 'anomaly'
                THEN 1
                ELSE 0
            END
        ) AS anomaly_count,

        SUM(
            CASE
                WHEN issue_type = 'invalid_reference'
                THEN 1
                ELSE 0
            END
        ) AS invalid_reference_count,

        SUM(
            CASE
                WHEN issue_type = 'missing_value'
                THEN 1
                ELSE 0
            END
        ) AS missing_value_count,

        SUM(
            CASE
                WHEN issue_type = 'duplicate_record'
                THEN 1
                ELSE 0
            END
        ) AS duplicate_issue_count

    FROM fact_data_quality;
    """,
    connection
)

data_quality_kpis

,total_issues,anomaly_count,invalid_reference_count,missing_value_count,duplicate_issue_count
0,13,1,1,5,2


In [71]:
total_operational_rows = (
    len(costs)
    + len(progress)
    + len(schedule)
    + len(procurement)
)

total_issues = int(
    data_quality_kpis["total_issues"].iloc[0]
)

data_quality_score = round(
    (
        1 - total_issues / total_operational_rows
    ) * 100,
    2
)

print(
    "Data Quality Score:",
    data_quality_score,
    "%"
)

Data Quality Score: 99.46 %


In [72]:
management_scorecard.to_sql(
    "report_management_scorecard",
    connection,
    if_exists="replace",
    index=False
)

print("Management scorecard saved to SQLite.")

Management scorecard saved to SQLite.


In [73]:
scorecard_check = pd.read_sql_query(
    """
    SELECT *
    FROM report_management_scorecard
    ORDER BY management_status, project_code;
    """,
    connection
)

scorecard_check

,project_code,project_name,approved_budget,actual_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,health_status,management_status,forecast_variance_pct,total_procurement_items,delayed_items,critical_delays
0,PRJ-002,Atlas Data Centre,42000000.0,41722937.17,44486768.69,-2486768.69,79.17,72.00,-7.17,Amber,Amber,-5.92,16,14,0
1,PRJ-005,Phoenix Industrial Facility,36000000.0,32138509.46,37672967.37,-1672967.37,72.73,70.25,-2.48,Green,Amber,-4.65,15,12,1
2,PRJ-001,Orion Commercial Tower,28000000.0,30496695.01,28429070.33,-429070.33,87.50,87.50,0.00,Green,Green,-1.53,13,9,0
3,PRJ-004,NorthLink Infrastructure,55000000.0,45922243.09,54095439.02,904560.98,71.88,71.79,-0.09,Green,Green,1.64,14,9,0
4,PRJ-007,WestGate Logistics Hub,24000000.0,21406361.89,23630422.69,369577.31,77.78,79.10,1.32,Green,Green,1.54,15,7,0
5,PRJ-003,Riverside Residential,19000000.0,24382525.02,22978606.83,-3978606.83,95.45,80.44,-15.01,Red,Red,-20.94,15,12,0
6,PRJ-006,St Anne Hospital Expansion,31000000.0,36138393.25,35198798.58,-4198798.58,90.91,76.28,-14.63,Red,Red,-13.54,14,14,1
7,PRJ-008,Helios Energy Facility,48000000.0,41553331.23,52471799.98,-4471799.98,69.23,64.96,-4.27,Amber,Red,-9.32,13,11,0


In [74]:
print(
    "Rows in management scorecard:",
    len(scorecard_check)
)

Rows in management scorecard: 8


In [90]:
# ---------------------------------------------------------
# BUILD PORTFOLIO KPI SUMMARY
# ---------------------------------------------------------

portfolio_summary = portfolio_kpis.copy()

portfolio_summary["forecast_variance"] = (
    portfolio_summary["total_budget"]
    - portfolio_summary["forecast_at_completion"]
)

portfolio_summary["planned_progress_pct"] = (
    portfolio_progress_summary["planned_progress_pct"]
)

portfolio_summary["actual_progress_pct"] = (
    portfolio_progress_summary["actual_progress_pct"]
)

portfolio_summary["progress_variance_pct"] = (
    portfolio_summary["actual_progress_pct"]
    - portfolio_summary["planned_progress_pct"]
)

portfolio_summary["data_quality_score"] = (
    data_quality_score
)

portfolio_summary

,total_budget,actual_cost_to_date,committed_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,data_quality_score
0,283000000.0,2.737610e+08,2.892594e+08,2.989639e+08,-15963873.49,78.33,73.71,-4.62,99.59


In [85]:
import sqlite3

connection = sqlite3.connect(DB_PATH)

print("Reconnected to:")
print(DB_PATH)

Reconnected to:
/Users/ansh/Desktop/ControlSight/data/database/controlsight.db


In [91]:
portfolio_summary.to_sql(
    "report_portfolio_kpis",
    connection,
    if_exists="replace",
    index=False
)

print("Portfolio KPI summary saved to SQLite.")

Portfolio KPI summary saved to SQLite.


In [92]:
portfolio_kpi_check = pd.read_sql_query(
    """
    SELECT *
    FROM report_portfolio_kpis;
    """,
    connection
)

portfolio_kpi_check

,total_budget,actual_cost_to_date,committed_cost_to_date,forecast_at_completion,forecast_variance,planned_progress_pct,actual_progress_pct,progress_variance_pct,data_quality_score
0,283000000.0,2.737610e+08,2.892594e+08,2.989639e+08,-15963873.49,78.33,73.71,-4.62,99.59


In [78]:
# ---------------------------------------------------------
# FINAL DATABASE AUDIT
# ---------------------------------------------------------

final_tables = [
    "dim_project",
    "dim_vendor",
    "dim_package",
    "dim_date",
    "fact_cost",
    "fact_progress",
    "fact_schedule",
    "fact_procurement",
    "fact_data_quality",
    "report_management_scorecard",
    "report_portfolio_kpis",
]

audit_rows = []

for table in final_tables:

    result = pd.read_sql_query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM {table};
        """,
        connection
    )

    audit_rows.append({
        "table": table,
        "row_count": int(result["row_count"].iloc[0]),
    })

final_database_audit = pd.DataFrame(audit_rows)

final_database_audit

,table,row_count
0,dim_project,8
1,dim_vendor,30
2,dim_package,96
3,dim_date,1461
4,fact_cost,1821
5,fact_progress,151
6,fact_schedule,318
7,fact_procurement,115
8,fact_data_quality,13
9,report_management_scorecard,8


In [79]:
foreign_key_check = pd.read_sql_query(
    """
    PRAGMA foreign_key_check;
    """,
    connection
)

foreign_key_check

,table,rowid,parent,fkid


In [80]:
connection.close()

print("Database connection closed successfully.")

Database connection closed successfully.


In [88]:
# ---------------------------------------------------------
# DATA QUALITY METHODOLOGY AUDIT
# ---------------------------------------------------------

RAW = ROOT / "data" / "raw"

raw_operational_files = {
    "cost_monthly": "cost_monthly.csv",
    "progress_updates": "progress_updates.csv",
    "schedule_activities": "schedule_activities.csv",
    "procurement_packages": "procurement_packages.csv",
}

clean_operational_frames = {
    "cost_monthly": costs,
    "progress_updates": progress,
    "schedule_activities": schedule,
    "procurement_packages": procurement,
}

audit_rows = []

for dataset, filename in raw_operational_files.items():

    raw_df = pd.read_csv(
        RAW / filename
    )

    clean_df = clean_operational_frames[
        dataset
    ]

    raw_rows = len(raw_df)
    accepted_rows = len(clean_df)
    excluded_rows = raw_rows - accepted_rows

    issue_count = int(
        (
            data_quality["dataset"]
            == dataset
        ).sum()
    )

    audit_rows.append({
        "dataset": dataset,
        "source_rows": raw_rows,
        "accepted_rows": accepted_rows,
        "excluded_rows": excluded_rows,
        "logged_issues": issue_count,
        "validation_pass_rate_pct": round(
            accepted_rows
            / raw_rows
            * 100,
            2
        ),
    })


dq_methodology_audit = pd.DataFrame(
    audit_rows
)


total_source_rows = int(
    dq_methodology_audit[
        "source_rows"
    ].sum()
)

total_accepted_rows = int(
    dq_methodology_audit[
        "accepted_rows"
    ].sum()
)

total_excluded_rows = (
    total_source_rows
    - total_accepted_rows
)

total_logged_issues = int(
    dq_methodology_audit[
        "logged_issues"
    ].sum()
)

validation_pass_rate = round(
    total_accepted_rows
    / total_source_rows
    * 100,
    2
)


print("DATA QUALITY METHODOLOGY AUDIT")
print("=" * 60)

display(
    dq_methodology_audit
)

print()
print(
    "Total source rows:",
    total_source_rows
)

print(
    "Total accepted rows:",
    total_accepted_rows
)

print(
    "Total excluded rows:",
    total_excluded_rows
)

print(
    "Total logged issues:",
    total_logged_issues
)

print(
    "Validation pass rate:",
    f"{validation_pass_rate}%"
)

print(
    "Current dashboard score:",
    f"{data_quality_score}%"
)

DATA QUALITY METHODOLOGY AUDIT


,dataset,source_rows,accepted_rows,excluded_rows,logged_issues,validation_pass_rate_pct
0,cost_monthly,1824,1821,3,4,99.84
1,progress_updates,152,151,1,2,99.34
2,schedule_activities,320,318,2,2,99.38
3,procurement_packages,119,115,4,5,96.64



Total source rows: 2415
Total accepted rows: 2405
Total excluded rows: 10
Total logged issues: 13
Validation pass rate: 99.59%
Current dashboard score: 99.46%


In [89]:
# ---------------------------------------------------------
# VALIDATION PASS RATE
# ---------------------------------------------------------
#
# Measures the proportion of source operational records
# admitted to the controlled analytical layer after ETL
# validation.
#
# Logged data-quality issues are NOT used as the numerator
# because:
#   1. one record may generate multiple issues; and
#   2. some suspicious but structurally valid records are
#      deliberately retained and flagged for investigation.
#
# Formula:
#
#     Accepted analytical records
#     --------------------------- × 100
#       Source records evaluated
# ---------------------------------------------------------

raw_operational_files = [
    ROOT / "data" / "raw" / "cost_monthly.csv",
    ROOT / "data" / "raw" / "progress_updates.csv",
    ROOT / "data" / "raw" / "schedule_activities.csv",
    ROOT / "data" / "raw" / "procurement_packages.csv",
]

total_source_rows = sum(
    len(pd.read_csv(file_path))
    for file_path in raw_operational_files
)

total_accepted_rows = (
    len(costs)
    + len(progress)
    + len(schedule)
    + len(procurement)
)

data_quality_score = round(
    (
        total_accepted_rows
        / total_source_rows
    ) * 100,
    2
)

print(
    "Validation Pass Rate:",
    data_quality_score,
    "%"
)

Validation Pass Rate: 99.59 %
